In [1]:
import pandas as pd
import requests

In [15]:

url = "https://odds.p.rapidapi.com/v4/sports/upcoming/odds"

querystring = {"regions":"us","oddsFormat":"decimal","markets":"h2h,spreads","dateFormat":"iso"}

headers = {
	"x-rapidapi-key": "e06deac331mshe504ceb9a75df65p1620e8jsn991187b549e3",
	"x-rapidapi-host": "odds.p.rapidapi.com",
	"Content-Type": "application/json"
}

response = requests.get(url, headers=headers, params=querystring)

print(response.json())

[{'id': '2b94791c4f5079ae37c12fb8743e8967', 'sport_key': 'baseball_npb', 'sport_title': 'NPB', 'commence_time': '2026-04-12T04:00:00Z', 'home_team': 'Hokkaido Nippon-Ham Fighters', 'away_team': 'Fukuoka SoftBank Hawks', 'bookmakers': [{'key': 'fanduel', 'title': 'FanDuel', 'last_update': '2026-04-12T06:43:40Z', 'markets': [{'key': 'h2h', 'last_update': '2026-04-12T06:43:40Z', 'outcomes': [{'name': 'Fukuoka SoftBank Hawks', 'price': 1.38}, {'name': 'Hokkaido Nippon-Ham Fighters', 'price': 3.0}]}, {'key': 'spreads', 'last_update': '2026-04-12T06:43:40Z', 'outcomes': [{'name': 'Fukuoka SoftBank Hawks', 'price': 2.46, 'point': -1.5}, {'name': 'Hokkaido Nippon-Ham Fighters', 'price': 1.52, 'point': 1.5}]}]}, {'key': 'fanatics', 'title': 'Fanatics', 'last_update': '2026-04-12T06:43:29Z', 'markets': [{'key': 'h2h', 'last_update': '2026-04-12T06:43:29Z', 'outcomes': [{'name': 'Fukuoka SoftBank Hawks', 'price': 1.36}, {'name': 'Hokkaido Nippon-Ham Fighters', 'price': 2.8}]}, {'key': 'spreads', 

In [16]:
data = response.json()

temp_df = pd.DataFrame([{
    
    # match info
    'sport_key': match.get('sport_key'),
    'commence_time': match.get('commence_time'),
    'home_team': match.get('home_team'),
    'away_team': match.get('away_team'),
    
    # 🔥 best odds (from all bookmakers)
    'home_team_odds': min([
        outcome['price']
        for bookmaker in match.get('bookmakers', [])
        for market in bookmaker.get('markets', [])
        if market.get('key') == 'h2h'
        for outcome in market.get('outcomes', [])
        if outcome.get('name') == match.get('home_team')
    ], default=None),
    
    'away_team_odds': min([
        outcome['price']
        for bookmaker in match.get('bookmakers', [])
        for market in bookmaker.get('markets', [])
        if market.get('key') == 'h2h'
        for outcome in market.get('outcomes', [])
        if outcome.get('name') == match.get('away_team')
    ], default=None),
    
   
    'home_point': next((
        outcome.get('point')
        for bookmaker in match.get('bookmakers', [])
        for market in bookmaker.get('markets', [])
        if market.get('key') == 'spreads'
        for outcome in market.get('outcomes', [])
        if outcome.get('name') == match.get('home_team')
    ), None),
    
    'away_point': next((
        outcome.get('point')
        for bookmaker in match.get('bookmakers', [])
        for market in bookmaker.get('markets', [])
        if market.get('key') == 'spreads'
        for outcome in market.get('outcomes', [])
        if outcome.get('name') == match.get('away_team')
    ), None),

} for match in data])

print(temp_df.head())

               sport_key         commence_time                     home_team  \
0           baseball_npb  2026-04-12T04:00:00Z  Hokkaido Nippon-Ham Fighters   
1           baseball_npb  2026-04-12T04:00:38Z           Saitama Seibu Lions   
2           baseball_npb  2026-04-12T04:01:00Z  Tohoku Rakuten Golden Eagles   
3           baseball_npb  2026-04-12T04:30:00Z              Chunichi Dragons   
4  soccer_korea_kleague1  2026-04-12T05:00:00Z               Daejeon Citizen   

                away_team  home_team_odds  away_team_odds  home_point  \
0  Fukuoka SoftBank Hawks            2.55            1.35         1.5   
1     Chiba Lotte Marines            5.00            1.01         1.5   
2          Orix Buffaloes            1.01           34.00        -3.5   
3          Hanshin Tigers           10.25            1.01         3.5   
4              Gangwon FC           20.00            1.10         NaN   

   away_point  
0        -1.5  
1        -1.5  
2         3.5  
3        -3.5  


In [18]:
temp_df

,sport_key,commence_time,home_team,away_team,home_team_odds,away_team_odds,home_point,away_point
0,baseball_npb,2026-04-12T04:00:00Z,Hokkaido Nippon-Ham Fighters,Fukuoka SoftBank Hawks,2.55,1.35,1.50,-1.50
1,baseball_npb,2026-04-12T04:00:38Z,Saitama Seibu Lions,Chiba Lotte Marines,5.00,1.01,1.50,-1.50
2,baseball_npb,2026-04-12T04:01:00Z,Tohoku Rakuten Golden Eagles,Orix Buffaloes,1.01,34.00,-3.50,3.50
3,baseball_npb,2026-04-12T04:30:00Z,Chunichi Dragons,Hanshin Tigers,10.25,1.01,3.50,-3.50
4,soccer_korea_kleague1,2026-04-12T05:00:00Z,Daejeon Citizen,Gangwon FC,20.00,1.10,NaN,NaN
5,soccer_australia_aleague,2026-04-12T05:00:00Z,Melbourne City,Wellington Phoenix FC,1.00,23.00,-1.25,1.25
6,soccer_japan_j_league,2026-04-12T05:00:00Z,Urawa Red Diamonds,Tokyo Verdy,3.60,5.25,NaN,NaN
7,baseball_kbo,2026-04-12T05:00:28Z,Hanwha Eagles,Kia Tigers,3.00,1.16,3.00,-3.00
8,baseball_kbo,2026-04-12T05:00:41Z,LG Twins,SSG Landers,1.01,9.50,-4.00,4.00
9,baseball_kbo,2026-04-12T05:00:46Z,KT Wiz,Doosan Bears,1.03,6.25,-3.00,3.00


In [19]:
df = pd.DataFrame()

In [20]:

df = pd.concat([df, temp_df], ignore_index=True)


In [21]:
df.head()

,sport_key,commence_time,home_team,away_team,home_team_odds,away_team_odds,home_point,away_point
0,baseball_npb,2026-04-12T04:00:00Z,Hokkaido Nippon-Ham Fighters,Fukuoka SoftBank Hawks,2.55,1.35,1.5,-1.5
1,baseball_npb,2026-04-12T04:00:38Z,Saitama Seibu Lions,Chiba Lotte Marines,5.00,1.01,1.5,-1.5
2,baseball_npb,2026-04-12T04:01:00Z,Tohoku Rakuten Golden Eagles,Orix Buffaloes,1.01,34.00,-3.5,3.5
3,baseball_npb,2026-04-12T04:30:00Z,Chunichi Dragons,Hanshin Tigers,10.25,1.01,3.5,-3.5
4,soccer_korea_kleague1,2026-04-12T05:00:00Z,Daejeon Citizen,Gangwon FC,20.00,1.10,NaN,NaN


In [22]:
df.shape

(23, 8)

In [24]:
df.to_csv('theoddsAPI.csv')